In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_PATH = "/content/ModelNet10_12views_224x224"

IMG_SIZE = 224
CHANNELS = 3
BATCH_SIZE = 64
EPOCHS = 30
VAL_RATIO = 0.15
SEED = 42
N_VIEWS = 12
PREPROCESSING = "mobilenet"
EXPERIMENT_NAME = "E2 MobileNetV2 pretrained 12 views 30 epochs"
BEST_MODEL_OUTPUT = "/content/drive/MyDrive/E2_MobileNetV2_12views_30epocas_BEST.keras"
FINAL_MODEL_OUTPUT = "/content/drive/MyDrive/E2_MobileNetV2_12views_30epocas_FINAL.keras"
HISTORY_OUTPUT = "/content/drive/MyDrive/E2_historico_MobileNetV2_12views_30epocas.csv"
RESULTS_OUTPUT = "/content/drive/MyDrive/E2_resultados_MobileNetV2_12views_30epocas.txt"
CONFUSION_MATRIX_OUTPUT = "/content/drive/MyDrive/E2_matriz_confusao_MobileNetV2_12views_30epocas.csv"


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.backend.clear_session()

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

In [ ]:
import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/ModelNet10_12views_224x224.zip"
EXTRACT_TO = "/content"

if not os.path.exists(DATASET_PATH):
    print("Dataset not found in /content. Extracting from Google Drive...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_TO)
else:
    print("Dataset already exists in /content.")

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Train exists:", os.path.exists(os.path.join(DATASET_PATH, "train")))
print("Test exists:", os.path.exists(os.path.join(DATASET_PATH, "test")))

if os.path.exists(DATASET_PATH):
    print("Dataset content:", os.listdir(DATASET_PATH))

In [ ]:
def get_classes(dataset_path):
    train_path = os.path.join(dataset_path, "train")
    classes = [
        d for d in os.listdir(train_path)
        if os.path.isdir(os.path.join(train_path, d))
    ]
    classes.sort()
    return classes


def load_objects(dataset_path, split, class_to_idx, n_views):
    split_path = os.path.join(dataset_path, split)
    objects = []

    for class_name, label in class_to_idx.items():
        class_path = os.path.join(split_path, class_name)

        if not os.path.isdir(class_path):
            continue

        object_folders = sorted([
            d for d in os.listdir(class_path)
            if os.path.isdir(os.path.join(class_path, d))
        ])

        for obj_folder in object_folders:
            obj_path = os.path.join(class_path, obj_folder)
            image_paths = sorted([
                os.path.join(obj_path, f)
                for f in os.listdir(obj_path)
                if f.lower().endswith(".png")
            ])

            if len(image_paths) != n_views:
                print(f"Warning: {obj_path} has {len(image_paths)} views and will be ignored.")
                continue

            objects.append({
                "object_path": obj_path,
                "image_paths": image_paths,
                "label": label,
                "class_name": class_name,
            })

    return objects


def split_train_val_objects(train_objects, val_ratio=0.15):
    train_objects = train_objects.copy()
    random.shuffle(train_objects)

    n_val = int(len(train_objects) * val_ratio)
    val_objects = train_objects[:n_val]
    train_objects = train_objects[n_val:]

    return train_objects, val_objects


def objects_to_image_samples(objects):
    image_paths = []
    labels = []

    for obj in objects:
        for img_path in obj["image_paths"]:
            image_paths.append(img_path)
            labels.append(obj["label"])

    return image_paths, labels


def load_image_default(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def load_image_mobilenet(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label


def create_tf_dataset(image_paths, labels, preprocessing="default", shuffle=True):
    image_paths = tf.constant(image_paths)
    labels = tf.constant(labels, dtype=tf.int32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=2048,
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    if preprocessing == "mobilenet":
        load_fn = load_image_mobilenet
    else:
        load_fn = load_image_default

    ds = ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def predict_object(model, object_data, preprocessing="default"):
    images = []

    for img_path in object_data["image_paths"]:
        image = tf.io.read_file(img_path)
        image = tf.image.decode_png(image, channels=3)
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
        image = tf.cast(image, tf.float32)

        if preprocessing == "mobilenet":
            image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        else:
            image = image / 255.0

        images.append(image)

    images = tf.stack(images, axis=0)
    probs = model.predict(images, verbose=0)
    mean_probs = np.mean(probs, axis=0)
    predicted_label = int(np.argmax(mean_probs))

    return predicted_label, mean_probs


def evaluate_multiview(model, test_objects, class_names, preprocessing, results_output, confusion_matrix_output, experiment_name):
    y_true = []
    y_pred = []

    for idx, obj in enumerate(test_objects, start=1):
        pred_label, _ = predict_object(model, obj, preprocessing=preprocessing)
        y_true.append(obj["label"])
        y_pred.append(pred_label)

        if idx % 50 == 0:
            print(f"Evaluating objects: {idx}/{len(test_objects)}")

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    print(experiment_name)
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(report)

    np.savetxt(confusion_matrix_output, cm, delimiter=",", fmt="%d")

    with open(results_output, "w", encoding="utf-8") as file:
        file.write(f"{experiment_name}\n")
        file.write(f"Accuracy:  {acc:.4f}\n")
        file.write(f"Precision: {precision:.4f}\n")
        file.write(f"Recall:    {recall:.4f}\n")
        file.write(f"F1-score:  {f1:.4f}\n\n")
        file.write("Classes:\n")
        for idx, class_name in enumerate(class_names):
            file.write(f"{idx}: {class_name}\n")
        file.write("\nClassification report:\n")
        file.write(report)
        file.write("\nConfusion matrix:\n")
        file.write(str(cm))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


In [ ]:
print("Checking dataset")
print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Train exists:", os.path.exists(os.path.join(DATASET_PATH, "train")))
print("Test exists:", os.path.exists(os.path.join(DATASET_PATH, "test")))

class_names = get_classes(DATASET_PATH)
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

print("Classes:", class_names)
print("Number of classes:", len(class_names))

train_objects_all = load_objects(DATASET_PATH, "train", class_to_idx, N_VIEWS)
test_objects = load_objects(DATASET_PATH, "test", class_to_idx, N_VIEWS)
train_objects, val_objects = split_train_val_objects(train_objects_all, val_ratio=VAL_RATIO)

train_image_paths, train_labels = objects_to_image_samples(train_objects)
val_image_paths, val_labels = objects_to_image_samples(val_objects)

print("Original training objects:", len(train_objects_all))
print("Training objects:", len(train_objects))
print("Validation objects:", len(val_objects))
print("Test objects:", len(test_objects))
print("Training images:", len(train_image_paths))
print("Validation images:", len(val_image_paths))

In [ ]:
def build_mobilenetv2_model(num_classes):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
        include_top=False,
        weights="imagenet",
    )

    base_model.trainable = False

    inputs = tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.40)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_mobilenetv2_model(num_classes=len(class_names))
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_OUTPUT,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        HISTORY_OUTPUT,
        append=False,
    ),
]

In [ ]:
train_ds = create_tf_dataset(
    train_image_paths,
    train_labels,
    preprocessing=PREPROCESSING,
    shuffle=True,
)

val_ds = create_tf_dataset(
    val_image_paths,
    val_labels,
    preprocessing=PREPROCESSING,
    shuffle=False,
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(FINAL_MODEL_OUTPUT)

best_model = tf.keras.models.load_model(BEST_MODEL_OUTPUT)

evaluate_multiview(
    model=best_model,
    test_objects=test_objects,
    class_names=class_names,
    preprocessing=PREPROCESSING,
    results_output=RESULTS_OUTPUT,
    confusion_matrix_output=CONFUSION_MATRIX_OUTPUT,
    experiment_name=EXPERIMENT_NAME,
)